# What did Berkshire do last quarter?

Diff a manager's two most recent 13F filings position by position:
what entered the book, what left, and what got resized.

Needs `pandas` and `matplotlib`, and a free API key in
`THREESPREAD_API_KEY` ([signup](https://3spread.com/auth/signup)).

In [1]:
import pandas as pd

from py3spread import Client

client = Client()
MANAGER_CIK = "1067983"  # berkshire hathaway

filings = client.institutional_holdings.list(filing_manager_cik=MANAGER_CIK, limit=2)["data"]
new_f, old_f = filings[0], filings[1]
print(f"{new_f['filer_name']}: {old_f['period_of_report']} -> {new_f['period_of_report']}")


def positions(filing_id):
    df = pd.DataFrame(client.institutional_holdings.iter_holdings(filing_id=filing_id))
    df["value_usd"] = pd.to_numeric(df["value_usd"])
    df["ssh_prnamt"] = pd.to_numeric(df["ssh_prnamt"])
    return df.groupby(["cusip", "name_of_issuer"], as_index=False).agg(
        shares=("ssh_prnamt", "sum"), value=("value_usd", "sum"))


new, old = positions(new_f["filing_id"]), positions(old_f["filing_id"])
diff = new.merge(old, on="cusip", how="outer", suffixes=("_new", "_old"))
diff["name"] = diff["name_of_issuer_new"].fillna(diff["name_of_issuer_old"])

BERKSHIRE HATHAWAY INC: 2025-12-31 -> 2026-03-31


## Entered and exited

In [2]:
entered = diff[diff["shares_old"].isna()]
exited = diff[diff["shares_new"].isna()]

print("new positions:")
print(entered[["name", "shares_new", "value_new"]].to_string(index=False)
      if len(entered) else "  none")
print("\nclosed positions:")
print(exited[["name", "shares_old", "value_old"]].to_string(index=False)
      if len(exited) else "  none")

new positions:
               name  shares_new    value_new
       ALPHABET INC   3585215.0 1028454775.0
DELTA AIR LINES INC  39809456.0 2646532635.0
          MACYS INC   3038355.0   54963842.0

closed positions:
                        name  shares_old    value_old
              AMAZON COM INC   2276000.0  525346320.0
    ATLANTA BRAVES HLDGS INC    115428.0    4553634.0
CHARTER COMMUNICATIONS INC N   1060882.0  221459118.0
                  DIAGEO PLC    227750.0   19647993.0
           DOMINOS PIZZA INC   3350000.0 1396347000.0
              HEICO CORP NEW   1294612.0  326798907.0
    LAMAR ADVERTISING CO NEW   1202410.0  152201058.0
      LIBERTY MEDIA CORP DEL   3018555.0  297357853.0
     MASTERCARD INCORPORATED   3986648.0 2275897610.0
                   POOL CORP   3068885.0  702007444.0
      UNITEDHEALTH GROUP INC   5039564.0 1663610472.0
                    VISA INC   8297460.0 2910002197.0
                ALLEGION PLC    780133.0  124212776.0
                     AON PLC  

## Biggest resizes (by share count, so price moves do not pollute it)

In [3]:
both = diff.dropna(subset=["shares_new", "shares_old"]).copy()
both["share_change_pct"] = (both["shares_new"] / both["shares_old"] - 1) * 100
moves = both[both["share_change_pct"].abs() > 0.5].sort_values("share_change_pct")
(moves[["name", "shares_old", "shares_new", "share_change_pct"]]
 .style.format({"shares_old": "{:,.0f}", "shares_new": "{:,.0f}",
                "share_change_pct": "{:+.1f}%"}).hide(axis="index"))

name,shares_old,shares_new,share_change_pct
CONSTELLATION BRANDS INC,"13,000,000","632,890",-95.1%
NUCOR CORP,"6,407,749","3,907,075",-39.0%
CHEVRON CORPORATION,"130,156,362","84,375,856",-35.2%
DAVITA INC,"31,759,065","30,100,585",-5.2%
LIBERTY LIVE HOLDINGS INC,"10,917,661","10,587,143",-3.0%
BANK AMERICA CORP,"517,295,934","513,624,165",-0.7%
LENNAR CORP,"180,980","237,703",+31.3%
LENNAR CORP,"7,050,950","10,099,642",+43.2%
NEW YORK TIMES CO MTN BE,"5,065,744","15,146,535",+199.0%
ALPHABET INC,"17,846,142","54,249,798",+204.0%


Unchanged share counts drop out automatically, so what remains is what
the manager actually traded. Positions can also be tracked continuously
via `client.changes.iter("institutional_holdings")` instead of waiting
to diff quarters.